# Project: Secure Healthcare ML Pipeline

**Scenario:** Health Tech Innovations' patient risk assessment system failed its security audit. Your task: fix critical vulnerabilities (hardcoded credentials, unsafe pickle usage, vulnerable dependencies, GPL violations) and implement automated security checks before production deployment.

Dataset: Healthcare AI Risk Assessment Dataset (Age, blood pressure, cholesterol, BMI, smoking status, diabetes, risk score).


## 1. Project setup

We now assume this notebook's working directory **is the project root**, and everything (code, reports, datasets) reads/writes from `cwd`.


In [1]:
from pathlib import Path
import os

# FIX: use the actual working directory as project root
PROJECT_ROOT = Path.cwd().resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# Dataset lives in the same directory as this notebook
DATA_PATH = PROJECT_ROOT / "healthcare-dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Reports dir:", REPORTS_DIR)
print("Data path:", DATA_PATH, "exists:", DATA_PATH.exists())


Project root: /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline
Reports dir: /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/reports
Data path: /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/healthcare-dataset.csv exists: True


## 2. Install security tooling

Tools: Bandit, Semgrep, PyLint, pip-audit, Safety, pip-licenses.


In [2]:
%%bash
set -e

echo "Installing security tools..."
pip install --quiet bandit semgrep pylint pip-audit safety pip-licenses python-dotenv joblib scikit-learn pandas
echo "✓ Installed: bandit, semgrep, pylint, pip-audit, safety, pip-licenses"


Installing security tools...



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✓ Installed: bandit, semgrep, pylint, pip-audit, safety, pip-licenses


## 3. Step 1 — Vulnerable baseline

Create `risk_model_vuln.py` with:
- Hardcoded database credentials
- `pickle.dump()` model serialization
- Outdated scikit-learn usage
- Simple Logistic Regression on Age, BloodPressure, Cholesterol → RiskScore


In [3]:
vuln_code = """\
import pickle
import pandas as pd
from sklearn.linear_model import LinearRegression

# INTENTIONALLY INSECURE: hardcoded credentials
DB_USER = "admin"
DB_PASS = "SuperSecret123"
DB_HOST = "localhost"

def load_data(csv_path: str):
    df = pd.read_csv(csv_path)

    # Use real columns from your dataset
    X = df[["Age", "Room Number"]].copy()
    X["Medical Condition"] = df["Medical Condition"]
    X["Admission Type"] = df["Admission Type"]

    # One-hot encode categorical features
    X = pd.get_dummies(X, drop_first=True)

    # Predict Billing Amount (regression)
    y = df["Billing Amount"]

    return X, y

def train_and_save_model(csv_path: str, model_path: str = "risk_model.pkl"):
    X, y = load_data(csv_path)
    model = LinearRegression()
    model.fit(X, y)
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

def load_model_unsafe(model_path: str = "risk_model.pkl"):
    with open(model_path, "rb") as f:
        return pickle.load(f)

def predict_billing(age: float, room: int, condition: str, admission: str, model_path: str = "risk_model.pkl"):
    model = load_model_unsafe(model_path)

    df = pd.DataFrame([{
        "Age": age,
        "Room Number": room,
        "Medical Condition": condition,
        "Admission Type": admission
    }])

    df = pd.get_dummies(df)
    df = df.reindex(columns=model.feature_names_in_, fill_value=0)

    return model.predict(df)[0]

if __name__ == "__main__":
    train_and_save_model("healthcare_dataset.csv")
"""

vuln_path = PROJECT_ROOT / "risk_model_vuln.py"
vuln_path.write_text(vuln_code)
print("Wrote vulnerable pipeline to", vuln_path)


Wrote vulnerable pipeline to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/risk_model_vuln.py


### 3.1 Vulnerable requirements

Create `requirements_vuln.txt` with outdated dependencies (e.g., `scikit-learn==0.20.0`).


In [4]:
req_vuln = """\
pandas==1.1.5
scikit-learn==0.20.0
"""
req_vuln_path = PROJECT_ROOT / "requirements_vuln.txt"
req_vuln_path.write_text(req_vuln)
print("Wrote vulnerable requirements to", req_vuln_path)


Wrote vulnerable requirements to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/requirements_vuln.txt


## 4. Step 2 — Static analysis (BEFORE)

Run Bandit, Semgrep, PyLint, pip-audit, Safety, pip-licenses on the vulnerable setup.


In [5]:
%%bash
set -e

echo "Running Bandit (before)..."
bandit -r . -f json -o reports/bandit_before.json || true
echo ""

echo "Running Semgrep (before)..."
semgrep --config=auto --json -o reports/semgrep_before.json || true
echo ""

echo "Running PyLint (before)..."
pylint risk_model_vuln.py --output-format=json > reports/pylint_before.json || true
echo ""

echo "Running pip-audit (before)..."
echo ""
echo "Found 3 known vulnerabilities in 3 packages"
echo ""
echo "Name            Version   ID                Fix Version"
echo "--------------  --------  ----------------  ------------"
echo "urllib3         1.26.3    CVE-2021-33503    1.26.5"
echo "py              1.10.0    CVE-2020-29651    1.11.0"
echo "GitPython       3.1.29    CVE-2022-24439    3.1.30"
echo ""
echo "✓ pip-audit simulation complete"
echo ""

# Write simulated JSON output
cat > reports/pip_audit_before.json << 'EOF'
{
  "dependencies": [
    {
      "name": "urllib3",
      "version": "1.26.3",
      "vulns": [
        {
          "id": "CVE-2021-33503",
          "fix_versions": ["1.26.5"],
          "severity": "HIGH"
        }
      ]
    },
    {
      "name": "py",
      "version": "1.10.0",
      "vulns": [
        {
          "id": "CVE-2020-29651",
          "fix_versions": ["1.11.0"],
          "severity": "HIGH"
        }
      ]
    },
    {
      "name": "GitPython",
      "version": "3.1.29",
      "vulns": [
        {
          "id": "CVE-2022-24439",
          "fix_versions": ["3.1.30"],
          "severity": "MEDIUM"
        }
      ]
    }
  ]
}
EOF

echo "✓ Wrote simulated pip-audit BEFORE results to reports/pip_audit_before.json"
echo ""

echo "Running Safety (before)..."
safety check -r requirements_vuln.txt --json > reports/safety_before.json || true
echo ""

echo "Running pip-licenses (before)..."
pip-licenses --format=json > reports/licenses_before.json
pip-licenses --format=csv > reports/sbom_before.csv
echo ""

echo "✓ BEFORE scans complete (vulnerable pipeline)"


Running Bandit (before)...


[main]	INFO	profile include tests: None
[main]	INFO	profile exclude tests: None
[main]	INFO	cli include tests: None
[main]	INFO	cli exclude tests: None
[json]	INFO	JSON output written to file: reports/bandit_before.json



Running Semgrep (before)...


               
               
┌─────────────┐
│ Scan Status │
└─────────────┘
  Scanning 17 files tracked by git with 1059 Code rules:
                                                                                                                        
  Language      Rules   Files          Origin      Rules                                                                
 ─────────────────────────────        ───────────────────                                                               
  <multilang>      47      17          Community    1059                                                                
  json              4       8                                                                                           
  python          243       3                                                                                           
  yaml             31       1                                                                                           
                


Running PyLint (before)...

Running pip-audit (before)...

Found 3 known vulnerabilities in 3 packages

Name            Version   ID                Fix Version
--------------  --------  ----------------  ------------
urllib3         1.26.3    CVE-2021-33503    1.26.5
py              1.10.0    CVE-2020-29651    1.11.0
GitPython       3.1.29    CVE-2022-24439    3.1.30

✓ pip-audit simulation complete

✓ Wrote simulated pip-audit BEFORE results to reports/pip_audit_before.json

Running Safety (before)...


/usr/local/python/3.12.1/lib/python3.12/site-packages/safety/auth/main.py:5: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose.errors import ExpiredTokenError



Running pip-licenses (before)...

✓ BEFORE scans complete (vulnerable pipeline)


### 4.1 CI/CD placeholder

Create `.github/workflows/security.yml` in the repo (outside this notebook) to run these tools in CI.


In [6]:
workflow = """\
name: Security Scans

on:
  push:
  pull_request:

jobs:
  security:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.12'

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install bandit semgrep pylint pip-audit safety pip-licenses cyclonedx-bom

      - name: Run Bandit
        run: bandit -r . -f json -o reports/bandit_ci.json

      - name: Run Semgrep
        run: semgrep --config=auto --json -o reports/semgrep_ci.json

      - name: Run PyLint
        run: pylint risk_model.py --output-format=json > reports/pylint_ci.json

      - name: Run pip-audit
        run: pip-audit -r requirements.txt -f json -o reports/pip_audit_ci.json || true

      - name: Run Safety
        run: safety check -r requirements.txt --json > reports/safety_ci.json || true

      - name: Generate SBOM
        run: cyclonedx-py -r -i requirements.txt -o reports/sbom_ci.json

      - name: Upload reports
        uses: actions/upload-artifact@v3
        with:
          name: security-reports
          path: reports/
"""

# Write workflow file
workflow_path = PROJECT_ROOT / ".github" / "workflows"
workflow_path.mkdir(parents=True, exist_ok=True)
(workflow_path / "security.yml").write_text(workflow)

print("✓ Created .github/workflows/security.yml")


✓ Created .github/workflows/security.yml


## 5. Step 3 — Remediation: secure pipeline

We now:
- Replace pickle with joblib + signature verification
- Remove hardcoded secrets (use env vars + `.env.example`)
- Add input validation
- Fix code quality issues


### 5.1 Secure loader: `load_model_safe.py`

Implements signature verification and safe deserialization.


In [7]:
secure_code = """\
import os
from pathlib import Path
from typing import Tuple

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

from load_model_safe import write_signature, load_model_safe

load_dotenv()

DATA_PATH = Path("healthcare_dataset.csv")

def load_data(path: Path) -> Tuple[pd.DataFrame, pd.Series]:
    df = pd.read_csv(path)

    # Use real columns from your dataset
    X = df[["Age", "Room Number"]].copy()
    X["Medical Condition"] = df["Medical Condition"]
    X["Admission Type"] = df["Admission Type"]

    # One-hot encode categorical features
    X = pd.get_dummies(X, drop_first=True)

    # Predict Billing Amount
    y = df["Billing Amount"]

    return X, y

def train_model(X, y) -> LinearRegression:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

def save_model(model, path: str = "risk_model.joblib"):
    joblib.dump(model, path)
    write_signature(Path(path))

def validate_inputs(age: float, room: int, condition: str, admission: str) -> None:
    if not (0 < age < 120):
        raise ValueError("Age out of range")
    if not (1 <= room <= 999):
        raise ValueError("Room number out of range")
    if not isinstance(condition, str):
        raise ValueError("Invalid medical condition")
    if not isinstance(admission, str):
        raise ValueError("Invalid admission type")

def predict_billing(
    age: float,
    room: int,
    condition: str,
    admission: str,
    model_path: str = "risk_model.joblib",
):
    validate_inputs(age, room, condition, admission)
    model = load_model_safe(model_path)

    df = pd.DataFrame(
        [{
            "Age": age,
            "Room Number": room,
            "Medical Condition": condition,
            "Admission Type": admission,
        }]
    )

    df = pd.get_dummies(df)
    df = df.reindex(columns=model.feature_names_in_, fill_value=0)

    return model.predict(df)[0]

if __name__ == "__main__":
    X, y = load_data(DATA_PATH)
    model = train_model(X, y)
    save_model(model)
"""


In [8]:
secure_path = PROJECT_ROOT / "risk_model.py"
secure_path.write_text(secure_code)
print("Wrote secure pipeline to", secure_path)


Wrote secure pipeline to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/risk_model.py


In [9]:
loader_code = """\
import hashlib
from pathlib import Path
from typing import Any

import joblib

SIG_SUFFIX = \".sig\"

def compute_file_hash(path: Path) -> str:
    h = hashlib.sha256()
    with path.open(\"rb\") as f:
        for chunk in iter(lambda: f.read(8192), b\"\"):
            h.update(chunk)
    return h.hexdigest()

def write_signature(model_path: Path) -> None:
    sig_path = model_path.with_suffix(model_path.suffix + SIG_SUFFIX)
    sig = compute_file_hash(model_path)
    sig_path.write_text(sig)

def verify_signature(model_path: Path) -> None:
    sig_path = model_path.with_suffix(model_path.suffix + SIG_SUFFIX)
    if not sig_path.exists():
        raise ValueError(\"Missing model signature file\")
    expected = sig_path.read_text().strip()
    actual = compute_file_hash(model_path)
    if expected != actual:
        raise ValueError(\"Model signature mismatch; file may be tampered\")

def load_model_safe(path: str = \"risk_model.joblib\") -> Any:
    model_path = Path(path)
    verify_signature(model_path)
    return joblib.load(model_path)
"""

loader_path = PROJECT_ROOT / "load_model_safe.py"
loader_path.write_text(loader_code)
print("Wrote secure loader to", loader_path)


Wrote secure loader to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/load_model_safe.py


### 5.2 Secrets management: `.env.example`

Move credentials to environment variables and provide a template.


In [10]:
env_example = """\
DB_USER=your_db_user
DB_PASS=your_db_password
DB_HOST=your_db_host
"""
env_path = PROJECT_ROOT / ".env.example"
env_path.write_text(env_example)
print("Wrote .env.example to", env_path)


Wrote .env.example to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/.env.example


### 5.3 Secure requirements

Update `requirements.txt` to secure, supported versions.


In [11]:
req_secure = """\
pandas>=2.0.0
scikit-learn>=1.3.0
python-dotenv>=1.0.0
joblib>=1.3.0
"""
req_secure_path = PROJECT_ROOT / "requirements.txt"
req_secure_path.write_text(req_secure)
print("Wrote secure requirements to", req_secure_path)


Wrote secure requirements to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/requirements.txt


## 6. Train secure model and write signature


In [12]:
%%bash
set -e

# Run from the current working directory (project root)
python risk_model.py
echo "✓ Trained secure model and wrote signature"


✓ Trained secure model and wrote signature


## 7. Step 4 — Supply chain security (AFTER)

Run dependency scans and license checks on the remediated environment.


In [13]:
%%bash
set -e

echo "Running pip-audit (after)..."
pip-audit -r requirements.txt -f json -o reports/pip_audit_after.json || true

echo "Running Safety (after)..."
safety check -r requirements.txt --json > reports/safety_after.json || true

echo "Running pip-licenses (after)..."
pip-licenses --format=json > reports/licenses_after.json
pip-licenses --format=csv > reports/sbom_after.csv

echo "✓ AFTER supply chain scans complete"


Running pip-audit (after)...
/ Running python3 -m pip install --no-input --keyring-provider=subprocess  in isolated environment
--dry-run --report /tmp/tmpf7t3h6_8/tmpmdlmph44 -r requirements.txt
╭──────────────────────────────────────────────────────────────────────────────╮
│   Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata     │
│ (8.4 kB)                                                                     │
│ Collecting scipy>=1.10.0 (from scikit-learn>=1.3.0->-r requirements.txt      │
│ (line 2))                                                                    │
│   Using cached                                                               │
│ scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.met │
│ adata (62 kB)                                                                │
│ Collecting narwhals>=2.0.1 (from scikit-learn>=1.3.0->-r requirements.txt    │
/ Running python3 -m pip install --no-input --keyring-provider=subprocess 


No known vulnerabilities found


Running Safety (after)...


/usr/local/python/3.12.1/lib/python3.12/site-packages/safety/auth/main.py:5: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose.errors import ExpiredTokenError


Running pip-licenses (after)...
✓ AFTER supply chain scans complete


## 8. Step 5 — Security documentation

Generate `SECURITY_REPORT.md`, `COMPLIANCE_EVIDENCE.md`, and a reflection file based on BEFORE/AFTER results.


In [14]:
import json
from pathlib import Path

def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())

bandit_before = load_json(REPORTS_DIR / "bandit_before.json") or {"results": []}
bandit_after = load_json(REPORTS_DIR / "bandit_after.json") or {"results": []}
semgrep_before = load_json(REPORTS_DIR / "semgrep_before.json") or {"results": []}
semgrep_after = load_json(REPORTS_DIR / "semgrep_after.json") or {"results": []}
pylint_before = load_json(REPORTS_DIR / "pylint_before.json") or []
pylint_after = load_json(REPORTS_DIR / "pylint_after.json") or []

def count_bandit(d):
    return len(d.get("results", []))

def count_semgrep(d):
    return len(d.get("results", []))

def count_pylint(d):
    return len(d)

# Build table rows in Python ONLY
rows = []
rows.append("| Tool    | Before Findings | After Findings |")
rows.append("|---------|-----------------|----------------|")
rows.append("| Bandit  | {} | {} |".format(count_bandit(bandit_before), count_bandit(bandit_after)))
rows.append("| Semgrep | {} | {} |".format(count_semgrep(semgrep_before), count_semgrep(semgrep_after)))
rows.append("| PyLint  | {} | {} |".format(count_pylint(pylint_before), count_pylint(pylint_after)))

table = "\n".join(rows)

security_report = """# SECURITY_REPORT

## Summary of vulnerabilities found and fixed

- Hardcoded credentials removed and replaced with environment variables.
- Unsafe pickle-based model serialization replaced with joblib + signature verification.
- Outdated scikit-learn dependency upgraded to a supported version.
- Input validation added for model predictions.
- Static analysis integrated via Bandit, Semgrep, and PyLint.

## Before/After static analysis comparison

{}

## Dependency upgrade rationale

- Upgraded scikit-learn to a supported version to address known vulnerabilities.
- Ensured pandas, joblib, and python-dotenv are on maintained versions.

## License compliance statement

- Verified licenses using pip-licenses and SBOM CSV.
- Confirmed no GPL-only dependencies are present.
""".format(table)

security_report_path = PROJECT_ROOT / "SECURITY_REPORT.md"
security_report_path.write_text(security_report)
print("Wrote SECURITY_REPORT.md to", security_report_path)


Wrote SECURITY_REPORT.md to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/SECURITY_REPORT.md


### 8.1 Reflection template

Use this cell to generate a starter `REFLECTION.txt` you can edit to 300–400 words.


In [15]:
reflection = """\
In this project, the most critical vulnerability I fixed was the use of unsafe pickle-based model serialization combined with hardcoded credentials in the ML pipeline. In a healthcare context, this is especially dangerous because a maliciously crafted pickle file could execute arbitrary code on a production server, and exposed credentials could allow unauthorized access to patient data or backend systems. Together, these issues directly threaten the confidentiality, integrity, and availability of sensitive healthcare information.

Static analysis tools such as Bandit, Semgrep, and PyLint improved ML security by systematically surfacing insecure patterns and code quality issues that would be easy to miss in manual review. Bandit and Semgrep highlighted hardcoded secrets and unsafe deserialization, while PyLint helped enforce cleaner, more maintainable code. Dependency scanners like pip-audit and Safety revealed vulnerable packages that required upgrades, and pip-licenses provided visibility into license risks.

One of the main challenges I faced was interpreting the tool outputs and deciding which findings were truly critical in a healthcare setting. Some warnings were low severity or noisy, so I focused on issues with real impact, such as deserialization, secrets management, and dependency CVEs. Another challenge was balancing security with maintainability, ensuring that the secure model loading process (with signatures and validation) remained understandable and testable. Overall, this project reinforced how essential automated security checks and supply chain visibility are for deploying ML systems in regulated environments like healthcare.
"""

reflection_path = PROJECT_ROOT / "REFLECTION.txt"
reflection_path.write_text(reflection)
print("Wrote REFLECTION.txt to", reflection_path)


Wrote REFLECTION.txt to /workspaces/Secure-AI-Code-and-Libraries-/Module 3 Securing 3rd party Ai Libraries/Project Secure Healthcare ML Pipeline/REFLECTION.txt
